In [1]:
import os
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

In [2]:
class CFG:

    # ==========================
    # Model
    # ==========================

    vocab_size = 45000

    d_model = 768
    num_heads = 12
    num_layers = 12
    d_ff = 3072

    max_seq_length = 1024

    dropout = 0.1


    # ==========================
    # Paths
    # ==========================

    base_checkpoint = "virgo_base_final.pt"

    dataset_path = "virgo_chat_v1_tokenized.bin"

    tokenizer_path = "virgo_tokenizer.json"

    output_dir = "./virgo_chat_v1_sft"


    # ==========================
    # Training
    # ==========================

    epochs = 3

    batch_size = 64

    gradient_accumulation = 2   # Effective batch = 128

    learning_rate = 2e-5

    weight_decay = 0.01

    warmup_ratio = 0.03

    grad_clip = 1.0

    num_workers = 8

    pin_memory = True

    persistent_workers = True

    prefetch_factor = 4

In [3]:
from tokenizers import Tokenizer


tokenizer = Tokenizer.from_file(
    CFG.tokenizer_path
)


print("=" * 50)
print("Tokenizer Loaded")
print("=" * 50)


print(
    "Vocab size:",
    tokenizer.get_vocab_size()
)


# Check required tokens

for token in [
    "<bos>",
    "<eos>",
    "<pad>",
    "<newline>",
    "<tab>"
]:

    token_id = tokenizer.token_to_id(token)

    print(
        f"{token:12s} : {token_id}"
    )

Tokenizer Loaded
Vocab size: 45000
<bos>        : 2
<eos>        : 3
<pad>        : 0
<newline>    : 4
<tab>        : 5


In [4]:
class RotaryEmbedding(nn.Module):
    def __init__(self, d_k, max_seq_length=2048, base=10000.0):
        super(RotaryEmbedding, self).__init__()

        assert d_k % 2 == 0, "Head dimension must be even for RoPE"

        inv_freq = 1.0 / (base ** (torch.arange(0, d_k, 2).float() / d_k))
        positions = torch.arange(max_seq_length).float()

        freqs = torch.outer(positions, inv_freq)

        self.register_buffer("cos", freqs.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", freqs.sin()[None, None, :, :], persistent=False)

    def forward(self, Q, K):
        seq_length = Q.size(-2)

        cos = self.cos[:, :, :seq_length, :].to(dtype=Q.dtype)
        sin = self.sin[:, :, :seq_length, :].to(dtype=Q.dtype)

        Q_even = Q[..., 0::2]
        Q_odd = Q[..., 1::2]

        K_even = K[..., 0::2]
        K_odd = K[..., 1::2]

        Q = torch.stack((Q_even * cos - Q_odd * sin, Q_even * sin + Q_odd * cos), dim=-1).flatten(-2)
        K = torch.stack((K_even * cos - K_odd * sin, K_even * sin + K_odd * cos), dim=-1).flatten(-2)

        return Q, K

In [5]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_seq_length):
        super(MultiHeadAttention, self).__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

        self.rope = RotaryEmbedding(self.d_k, max_seq_length)

    def split_heads(self, x):
        batch_size, seq_length, _ = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        batch_size, _, seq_length, _ = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, x):
        Q = self.split_heads(self.W_q(x))
        K = self.split_heads(self.W_k(x))
        V = self.split_heads(self.W_v(x))

        Q, K = self.rope(Q, K)

        attn_output = F.scaled_dot_product_attention(Q, K, V, dropout_p=0.0, is_causal=True)

        return self.W_o(self.combine_heads(attn_output))

In [6]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()

        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.gelu = nn.GELU()

    def forward(self, x):
        return self.fc2(self.gelu(self.fc1(x)))

In [7]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, max_seq_length, dropout):
        super(TransformerBlock, self).__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads, max_seq_length)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        norm_x = self.norm1(x)
        x = x + self.dropout(self.self_attn(norm_x))

        norm_x = self.norm2(x)
        x = x + self.dropout(self.feed_forward(norm_x))

        return x

In [8]:
class VirgoModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff, max_seq_length, dropout):
        super(VirgoModel, self).__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_seq_length = max_seq_length

        self.token_embedding = nn.Embedding(vocab_size, d_model)

        self.layers = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, max_seq_length, dropout)
            for _ in range(num_layers)
        ])

        self.norm = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.dropout = nn.Dropout(dropout)

        self.lm_head.weight = self.token_embedding.weight

    def forward(self, input_ids):
        seq_length = input_ids.size(1)

        if seq_length > self.max_seq_length:
            raise ValueError(f"Sequence length {seq_length} exceeds maximum {self.max_seq_length}")

        x = self.token_embedding(input_ids)
        x = self.dropout(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)

        return self.lm_head(x)

In [9]:
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Using:", device)


model = VirgoModel(
    vocab_size=CFG.vocab_size,
    d_model=CFG.d_model,
    num_heads=CFG.num_heads,
    num_layers=CFG.num_layers,
    d_ff=CFG.d_ff,
    max_seq_length=CFG.max_seq_length,
    dropout=CFG.dropout
).to(device)


print("Virgo-Base architecture created")

Using: cuda
Virgo-Base architecture created


In [11]:
checkpoint = torch.load(
    CFG.base_checkpoint,
    map_location=device
)


print(
    "Checkpoint type:",
    type(checkpoint)
)

Checkpoint type: <class 'dict'>


In [13]:
checkpoint = torch.load(
    CFG.base_checkpoint,
    map_location=device
)

print(type(checkpoint))

if isinstance(checkpoint, dict):
    print(checkpoint.keys())

<class 'dict'>
dict_keys(['epoch', 'step', 'tokens_seen', 'model_state_dict', 'optimizer_state_dict', 'best_val_loss', 'cumulative_correct', 'cumulative_total', 'config'])


In [14]:
id="6_load_weights"
model.load_state_dict(
    checkpoint["model_state_dict"]
)


model.to(device)

model.eval()


print("✅ Virgo-Base weights loaded successfully")

✅ Virgo-Base weights loaded successfully


In [15]:
data = torch.load(
    CFG.dataset_path,
    map_location="cpu"
)


print(type(data))

if isinstance(data, dict):
    print(data.keys())

<class 'list'>


In [16]:
class VirgoChatDataset(torch.utils.data.Dataset):

    def __init__(self, data):

        self.samples = data

        self.virgo_ids = tokenizer.encode(
            "Virgo"
        ).ids


    def __len__(self):

        return len(self.samples)


    def __getitem__(self, idx):

        tokens = self.samples[idx]

        tokens = tokens[:CFG.max_seq_length]


        input_ids = torch.tensor(
            tokens[:-1],
            dtype=torch.long
        )


        labels = torch.tensor(
            tokens[1:],
            dtype=torch.long
        )


        # Find Virgo position
        start = -1

        for i in range(
            len(input_ids) - len(self.virgo_ids)
        ):

            if (
                input_ids[i:i+len(self.virgo_ids)].tolist()
                == self.virgo_ids
            ):
                start = i
                break


        # Mask user + "Virgo" tokens
        if start != -1:

            labels[
                :start + len(self.virgo_ids)
            ] = -100


        return {
            "input_ids": input_ids,
            "labels": labels
        }

In [17]:
train_dataset = VirgoChatDataset(
    data
)

print(
    "Dataset samples:",
    len(train_dataset)
)

Dataset samples: 1461523


In [18]:
sample = train_dataset[0]

print(
    sample["labels"][:100]
)

tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100])


In [19]:
labels = sample["labels"]

valid_positions = (
    labels != -100
).nonzero(as_tuple=True)[0]


print("First train position:")
print(valid_positions[0])


print("\nLabels around start:")
print(
    labels[
        valid_positions[0]-5:
        valid_positions[0]+20
    ]
)

First train position:
tensor(144)

Labels around start:
tensor([ -100,  -100,  -100,  -100,  -100, 18558,   294,   215,  6638,    17,
         1777,   215,  2959, 25096,   243,  5421,   233,   215, 19126, 14842,
          233,  3783,    17,  1179,   215])


In [44]:
from torch.nn.utils.rnn import pad_sequence
import torch

def collate_fn(batch):
    input_ids = [item["input_ids"] for item in batch]
    labels = [item["labels"] for item in batch]

    input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=0,      # <pad> token id
    )

    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100,   # ignore in loss
    )

    return {
        "input_ids": input_ids,
        "labels": labels,
    }

In [45]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=CFG.num_workers,
    pin_memory=True,
    persistent_workers=(CFG.num_workers > 0),
)

In [46]:
virgo_tokens = tokenizer.encode("Virgo").ids

print(virgo_tokens)

[59, 299, 2296]


In [47]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.learning_rate,
    weight_decay=CFG.weight_decay
)


print("Optimizer created")

Optimizer created


In [48]:
scaler = torch.amp.GradScaler(
    "cuda"
)


criterion = nn.CrossEntropyLoss(
    ignore_index=-100
)


print("AMP and loss ready")

AMP and loss ready


In [49]:
sample = train_dataset[0]

print(type(sample))
print(sample)

if isinstance(sample, dict):
    for k, v in sample.items():
        print(k, type(v))

<class 'dict'>
{'input_ids': tensor([    2,     4, 12187,   217,     4, 31614,  3033,   215,  1546,  1984,
           31,   283,  1271,   244,  1052,   215,  2250,   233,   980,   244,
          981,   212,  6638,   243,  3033,  2133,   538,   319,    19, 13684,
          317,  4504,  1583,   577,   314,   212,  1012,  1984,   289,   215,
         6638,   522,  3391, 19126,    17,   215, 14842,   233,  3783,   376,
          215,   923,   362,   448,   215,  2464,  4047,  2395,  1962,    18,
           76, 33346,  8612,   233,  1036,  2123,  9786,    17, 17913,    17,
         2123,  3783,    17, 27041,    17, 11043,   243,  4000,  3783,  3708,
          501,  2941,    19,   924,  4655,   215,  1723,   233,  1397,   293,
          452,   631,  2076,   303,  4047,  8612,   496,   508,   656,   397,
          212,  3363,  1397,   293,   376,  3432,   289,  6553,  7292,   215,
         2134,  2941,    19,  1112,   215,  5510,   233,  3783,  2164,   607,
         2717,    17,   215,  8612,

In [50]:
import torch
import torch.nn.functional as F

model.eval()

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)

with torch.no_grad():
    logits = model(input_ids)

print("Input shape :", input_ids.shape)
print("Labels shape:", labels.shape)
print("Logits shape:", logits.shape)

Input shape : torch.Size([64, 378])
Labels shape: torch.Size([64, 378])
Logits shape: torch.Size([64, 378, 45000])


In [51]:
model.train()


batch = next(iter(train_loader))


input_ids = batch["input_ids"].to(device)
labels = batch["labels"].to(device)


optimizer.zero_grad()


with torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):

    logits = model(
        input_ids
    )


loss = criterion(
    logits.view(-1, CFG.vocab_size),
    labels.view(-1)
)


scaler.scale(loss).backward()


scaler.step(
    optimizer
)

scaler.update()


print(
    "Loss:",
    loss.item()
)

Loss: 3.228515625


In [52]:
CFG.epochs

3

In [53]:
from tqdm.auto import tqdm
import time
import os


# =========================
# Setup
# =========================

os.makedirs(
    CFG.output_dir,
    exist_ok=True
)


best_accuracy = 0.0

global_step = 0

tokens_seen = 0

cumulative_correct = 0
cumulative_total = 0


model.train()



# =========================
# Training
# =========================

for epoch in range(CFG.epochs):


    epoch_loss = 0.0

    epoch_correct = 0
    epoch_total = 0


    epoch_start = time.time()


    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{CFG.epochs}"
    )


    optimizer.zero_grad()



    for step, batch in enumerate(progress):


        input_ids = batch["input_ids"].to(
            device,
            non_blocking=True
        )


        labels = batch["labels"].to(
            device,
            non_blocking=True
        )



        # count trainable tokens only

        batch_tokens = (
            labels != -100
        ).sum().item()


        tokens_seen += batch_tokens



        # =====================
        # Forward
        # =====================

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):


            logits = model(
                input_ids
            )


            loss = criterion(
                logits.view(
                    -1,
                    CFG.vocab_size
                ),
                labels.view(-1)
            )


            loss_scaled = (
                loss /
                CFG.gradient_accumulation
            )



        # =====================
        # Backward
        # =====================

        scaler.scale(
            loss_scaled
        ).backward()



        # =====================
        # Accuracy
        # =====================

        predictions = torch.argmax(
            logits,
            dim=-1
        )


        mask = (
            labels != -100
        )


        correct = (
            predictions[mask]
            ==
            labels[mask]
        ).sum().item()


        total = mask.sum().item()



        epoch_correct += correct
        epoch_total += total


        cumulative_correct += correct
        cumulative_total += total



        # =====================
        # Optimizer step
        # =====================

        if (
            (step + 1)
            %
            CFG.gradient_accumulation
            ==
            0
        ):


            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                CFG.grad_clip
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            optimizer.zero_grad()


            global_step += 1



        epoch_loss += loss.item()



        # =====================
        # Metrics
        # =====================

        elapsed = (
            time.time()
            -
            epoch_start
        )


        tok_per_sec = (
            tokens_seen /
            max(elapsed,1)
        )


        current_accuracy = (
            epoch_correct /
            max(epoch_total,1)
        )


        cumulative_accuracy = (
            cumulative_correct /
            max(cumulative_total,1)
        )


        current_lr = (
            optimizer
            .param_groups[0]["lr"]
        )



        progress.set_postfix(
            {

                "loss":
                f"{loss.item():.3f}",


                "acc":
                f"{current_accuracy*100:.2f}%",


                "cum_acc":
                f"{cumulative_accuracy*100:.2f}%",


                "lr":
                f"{current_lr:.2e}",


                "tok/s":
                f"{tok_per_sec:.0f}",


                "tokens":
                f"{tokens_seen/1e6:.2f}M"

            }
        )



    # =========================
    # Epoch summary
    # =========================


    avg_loss = (
        epoch_loss /
        len(train_loader)
    )


    epoch_accuracy = (
        epoch_correct /
        max(epoch_total,1)
    )


    cumulative_accuracy = (
        cumulative_correct /
        max(cumulative_total,1)
    )


    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    print("\n")
    print("="*40)

    print(
        f"Epoch              : {epoch+1}/{CFG.epochs}"
    )

    print(
        f"Loss               : {avg_loss:.4f}"
    )

    print(
        f"Epoch Accuracy      : {epoch_accuracy*100:.2f}%"
    )

    print(
        f"Cumulative Accuracy : {cumulative_accuracy*100:.2f}%"
    )

    print(
        f"Learning Rate       : {current_lr:.2e}"
    )

    print(
        f"Tokens Seen         : {tokens_seen:,}"
    )

    print(
        f"Global Steps        : {global_step}"
    )

    print("="*40)



    # =========================
    # Save checkpoint
    # =========================


    checkpoint = {

        "epoch":
        epoch + 1,


        "step":
        global_step,


        "tokens_seen":
        tokens_seen,


        "epoch_accuracy":
        epoch_accuracy,


        "cumulative_accuracy":
        cumulative_accuracy,


        "model_state_dict":
        model.state_dict(),


        "optimizer_state_dict":
        optimizer.state_dict(),

    }



    # Save every epoch

    torch.save(
        checkpoint,
        f"{CFG.output_dir}/virgo_chat_epoch_{epoch+1}.pt"
    )



    # Save best

    if epoch_accuracy > best_accuracy:


        best_accuracy = epoch_accuracy


        torch.save(
            checkpoint,
            f"{CFG.output_dir}/virgo_chat_best.pt"
        )


        print(
            "🔥 New best Virgo-Chat model saved!"
        )



print("\n✅ Virgo-Chat training finished")

Epoch 1/3:   0%|          | 0/22837 [00:00<?, ?it/s]



Epoch              : 1/3
Loss               : 1.4983
Epoch Accuracy      : 65.06%
Cumulative Accuracy : 65.06%
Learning Rate       : 2.00e-05
Tokens Seen         : 172,836,945
Global Steps        : 11418
🔥 New best Virgo-Chat model saved!


Epoch 2/3:   0%|          | 0/22837 [00:00<?, ?it/s]



Epoch              : 2/3
Loss               : 1.2990
Epoch Accuracy      : 68.31%
Cumulative Accuracy : 66.68%
Learning Rate       : 2.00e-05
Tokens Seen         : 345,673,890
Global Steps        : 22836
🔥 New best Virgo-Chat model saved!


Epoch 3/3:   0%|          | 0/22837 [00:00<?, ?it/s]



Epoch              : 3/3
Loss               : 1.2196
Epoch Accuracy      : 69.67%
Cumulative Accuracy : 67.68%
Learning Rate       : 2.00e-05
Tokens Seen         : 518,510,835
Global Steps        : 34254
🔥 New best Virgo-Chat model saved!

✅ Virgo-Chat training finished


In [54]:
import torch


device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)


checkpoint = torch.load(
    f"{CFG.output_dir}/virgo_chat_best.pt",
    map_location=device
)


model.load_state_dict(
    checkpoint["model_state_dict"]
)


model.to(device)

model.eval()


print("✅ Virgo-Chat model loaded")

✅ Virgo-Chat model loaded


In [55]:
def generate_virgo(
    prompt,
    max_new_tokens=200,
    temperature=0.7,
    top_k=50
):

    model.eval()


    formatted_prompt = (
        "<bos>"
        "<newline>User<newline>"
        + prompt
        + "<newline><newline>"
        + "Virgo<newline>"
    )


    input_ids = tokenizer.encode(
        formatted_prompt
    ).ids


    input_ids = torch.tensor(
        input_ids,
        dtype=torch.long
    ).unsqueeze(0).to(device)



    eos_id = tokenizer.token_to_id(
        "<eos>"
    )


    with torch.no_grad():

        for _ in range(max_new_tokens):


            logits = model(
                input_ids
            )


            logits = logits[:, -1, :]



            # temperature

            logits = (
                logits /
                temperature
            )



            # top-k sampling

            if top_k is not None:

                values, indices = torch.topk(
                    logits,
                    top_k
                )


                filtered_logits = torch.full_like(
                    logits,
                    float("-inf")
                )


                filtered_logits.scatter_(
                    1,
                    indices,
                    values
                )


                logits = filtered_logits



            probs = torch.softmax(
                logits,
                dim=-1
            )


            next_token = torch.multinomial(
                probs,
                num_samples=1
            )


            input_ids = torch.cat(
                [
                    input_ids,
                    next_token
                ],
                dim=1
            )


            if next_token.item() == eos_id:

                break



    output = tokenizer.decode(
        input_ids[0].tolist()
    )


    return output

In [56]:
id="clean_response"
def clean_response(text):

    # remove everything before Virgo
    if "Virgo" in text:

        text = text.split(
            "Virgo"
        )[-1]


    # remove eos

    if "<eos>" in text:

        text = text.split(
            "<eos>"
        )[0]


    # remove extra special tokens

    text = text.replace(
        "<newline>",
        "\n"
    )


    text = text.strip()


    return text

In [57]:
import time
import sys


def type_writer(
    text,
    delay=0.02
):

    for char in text:

        sys.stdout.write(char)

        sys.stdout.flush()

        time.sleep(delay)

    print()

In [ ]:
prompt = input("User: ")


raw_response = generate_virgo(
    prompt,
    max_new_tokens=200,
    temperature=0.7,
    top_k=50
)


response = clean_response(
    raw_response
)


print("\nVirgo:\n")


type_writer(
    response,
    delay=0.02
)

User:  hi
